<a href="https://colab.research.google.com/github/lax-17/pose-guided-diffusion-analysis/blob/main/notebooks/Pose_Fidelity_measured_using_AP%2C_PCK%2C_and_OKS_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# BLOCK 1: INSTALL EVALUATION TOOLS
!pip install -q ultralytics scipy torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 66.6 MB/s eta 0:00:00


In [ ]:
# BLOCK 2: DEFINE METRICS (PCK & OKS)
import numpy as np

# COCO 'Sigmas' (Variances) for OKS calculation
# Controls how strict we are for different parts (e.g., eyes are stricter than hips)
OKS_SIGMAS = np.array([
    .026, .025, .025, .035, .035, .079, .079, .072, .072, .062, .062,
    .107, .107, .087, .087, .089, .089
]) / 10.0

def compute_oks(gt_kpts, pred_kpts, area):
    """
    Object Keypoint Similarity (OKS).
    gt_kpts: [17, 3] (x, y, v)
    pred_kpts: [17, 3] (x, y, conf)
    area: Scale of the person (w * h)
    """
    # Filter visible keypoints
    visible = gt_kpts[:, 2] > 0
    if np.sum(visible) == 0: return 0.0

    # Calculate distances
    d = np.linalg.norm(gt_kpts[visible, :2] - pred_kpts[visible, :2], axis=1)

    # Calculate OKS (The standard COCO formula)
    # Vars represents the "allowable margin of error" based on head size/body area
    vars = (2 * OKS_SIGMAS[visible])**2
    return np.mean(np.exp(-d**2 / (2 * area * vars + 1e-9)))

def compute_pck(gt_kpts, pred_kpts, threshold=0.1):
    """
    Percentage of Correct Keypoints (PCK).
    threshold: Fraction of bounding box size (default 0.1 = 10% of torso size)
    """
    visible = gt_kpts[:, 2] > 0
    if np.sum(visible) == 0: return 0.0

    # Calculate Torso Diameter (L-Shoulder to R-Hip) as scale reference
    # Indices: 5=L_Sh, 12=R_Hip (COCO format)
    if gt_kpts[5, 2] > 0 and gt_kpts[12, 2] > 0:
        scale = np.linalg.norm(gt_kpts[5, :2] - gt_kpts[12, :2])
    else:
        # Fallback: Max dimension of the skeleton
        scale = np.max(gt_kpts[:, :2]) - np.min(gt_kpts[:, :2])

    dist = np.linalg.norm(gt_kpts[visible, :2] - pred_kpts[visible, :2], axis=1)

    # Pass/Fail check
    correct = dist < (scale * threshold)
    return np.mean(correct)

print("Metric functions defined.")

✅ Metric functions defined.


In [ ]:
# BLOCK 3: RUN FIDELITY CHECK (CORRECTED)
from ultralytics import YOLO
from diffusers import StableDiffusionAdapterPipeline, T2IAdapter, UniPCMultistepScheduler
import torch
import pandas as pd
from PIL import Image
import os
import cv2
import numpy as np

# --- CONFIG ---
DEVICE = "cuda"
ADAPTER_PATH = "/content/drive/MyDrive/t2i_adapter_humanart_v1/checkpoint-step-15000"
PARQUET_PATH = "/content/drive/MyDrive/pose-guided-diffusion-analysis/processed_data/full_humanart_dataset.parquet"
IMAGE_ROOT = "/content/drive/MyDrive/pose-guided-diffusion-analysis/datasets_unzipped/HumanArt_v1"
NUM_TESTS = 50

# --- HELPER 1: POSE DRAWER (Re-defined here to avoid errors) ---
def draw_pose(img_h, img_w, keypoints_list):
    canvas = np.zeros((img_h, img_w, 3), dtype=np.uint8)
    edges = [(0,1), (0,2), (1,3), (2,4), (5,6), (5,7), (7,9), (6,8), (8,10),
             (5,11), (6,12), (11,12), (11,13), (13,15), (12,14), (14,16)]
    colors = [[255, 0, 0], [255, 85, 0], [255, 170, 0], [255, 255, 0],
              [170, 255, 0], [85, 255, 0], [0, 255, 0], [0, 255, 85],
              [0, 255, 170], [0, 255, 255], [0, 170, 255], [0, 85, 255],
              [0, 0, 255], [85, 0, 255], [170, 0, 255], [255, 0, 255]]

    # Ensure keypoints_list is a list of lists (for multiple people)
    if not isinstance(keypoints_list, list):
        return Image.fromarray(canvas)

    for person in keypoints_list:
        if person is None or len(person) < 51: continue
        kpts = np.array(person).reshape(-1, 3)

        # Draw Limbs
        for i, (s, e) in enumerate(edges):
            if s < len(kpts) and e < len(kpts) and kpts[s][2]>0 and kpts[e][2]>0:
                cv2.line(canvas, (int(kpts[s][0]), int(kpts[s][1])),
                         (int(kpts[e][0]), int(kpts[e][1])), colors[i%len(colors)], 3)
        # Draw Points
        for x, y, v in kpts:
            if v > 0: cv2.circle(canvas, (int(x), int(y)), 4, (255, 255, 255), -1)

    return Image.fromarray(canvas)

# --- HELPER 2: SCALE KEYPOINTS ---
def scale_keypoints(kpts, orig_w, orig_h, target_s=512):
    kpts = np.array(kpts).reshape(-1, 3)
    s_x = target_s / orig_w
    s_y = target_s / orig_h
    kpts[:, 0] *= s_x
    kpts[:, 1] *= s_y
    return kpts

# 1. Load Models
print("Loading Models...")
adapter = T2IAdapter.from_pretrained(ADAPTER_PATH, torch_dtype=torch.float16).to(DEVICE)
pipe = StableDiffusionAdapterPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", adapter=adapter, torch_dtype=torch.float16, safety_checker=None
).to(DEVICE)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pose_model = YOLO('yolov8n-pose.pt')

# 2. Prepare Data
df = pd.read_parquet(PARQUET_PATH)
samples = df.sample(NUM_TESTS, random_state=42)

pck_scores = []
oks_scores = []

print(f"Benchmarking Pose Fidelity on {NUM_TESTS} images...")

for i, (_, row) in enumerate(samples.iterrows()):
    try:
        # Load Original Image Info
        rel_path = row['relative_path'].replace("HumanArt/", "", 1) if row['relative_path'].startswith("HumanArt/") else row['relative_path']
        img_path = os.path.join(IMAGE_ROOT, rel_path)
        with Image.open(img_path) as orig_img:
            w, h = orig_img.size

        # Prepare Ground Truth Pose Map
        # Note: We wrap row['keypoints'] in a list [] because our drawer expects a list of people
        pose_img = draw_pose(512, 512, [row['keypoints']])

        # B. Generate Image
        generated_img = pipe(row['description'], image=pose_img, num_inference_steps=30, adapter_conditioning_scale=1.0).images[0]

        # C. Detect Pose (The Judge)
        results = pose_model(generated_img, verbose=False)

        # D. Compare
        gt_kpts = scale_keypoints(row['keypoints'], w, h, 512)

        if len(results[0].keypoints) > 0:
            pred_kpts = results[0].keypoints.data[0].cpu().numpy()

            # Area estimate for OKS
            area = (np.max(gt_kpts[:,0])-np.min(gt_kpts[:,0])) * (np.max(gt_kpts[:,1])-np.min(gt_kpts[:,1]))

            oks = compute_oks(gt_kpts, pred_kpts, area) # Ensure compute_oks is defined in Block 2!
            pck = compute_pck(gt_kpts, pred_kpts)       # Ensure compute_pck is defined in Block 2!

            oks_scores.append(oks)
            pck_scores.append(pck)
        else:
            oks_scores.append(0.0)
            pck_scores.append(0.0)

    except Exception as e:
        print(f"Skip {i}: {e}")

# 3. Final Report
print("\n" + "="*40)
print("POSE FIDELITY RESULTS")
print("="*40)
print(f"Sample Size: {NUM_TESTS}")
print(f"Average OKS: {np.mean(oks_scores):.4f}")
print(f"Average PCK: {np.mean(pck_scores):.4f}")
print("="*40)

⏳ Loading Models...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.t2i_adapter.pipeline_stable_diffusion_adapter.StableDiffusionAdapterPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


🚀 Benchmarking Pose Fidelity on 50 images...


  0%|          | 0/30 [00:00<?, ?it/s]

...

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]


🏆 POSE FIDELITY RESULTS
Sample Size: 50
Average OKS: 0.0015
Average PCK: 0.0065
